In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt


import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
gpus = jax.devices()
print(gpus)

jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from haiku import PRNGSequence

from dmpe.data_management import DataPaths
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results
from dmpe.models.models import NeuralEulerODECartpole

---

In [ ]:
from dmpe.utils.env_utils.fluid_tank_utils import setup_env as setup_fluid_tank_env

In [ ]:
env, penalty_function, featurize, _ = setup_fluid_tank_env()

In [ ]:
def recursive_feasibility_fluid_tank(h_k, u_k, tau, h_max, u_max, static_params):
    valid_state = jnp.logical_and(h_k <= h_max, h_k >= 0)
    valid_action = jnp.logical_and(u_k <= u_max, u_k >=0)

    valid = jnp.logical_and(valid_state, valid_action)

    valid_action_next_step_upper = (
        u_k <= (
            static_params.base_area / tau * (h_max - h_k) 
            + static_params.c_d * static_params.orifice_area * jnp.sqrt(2 * static_params.g * h_k)
        )
    )

    valid_action_next_step = jnp.logical_and(valid_action_next_step_upper, u_k >= 0)

    valid = jnp.logical_and(valid, valid_action_next_step)

    return valid

In [ ]:
h_max=env.env_properties.physical_normalizations.height.max
u_max=env.env_properties.action_normalizations.inflow.max

In [ ]:
recursive_feasibility_parameterized = eqx.filter_jit(
    partial(
        recursive_feasibility_fluid_tank,
        tau=env.tau,
        h_max=h_max,
        u_max=u_max,
        static_params=env.env_properties.static_params
    )
)
recursive_feasibility_parameterized

In [ ]:
heights_ = jnp.linspace(0, h_max, 250)
inflows_ = jnp.linspace(0, u_max, 250)

static_params = env.env_properties.static_params
allowed_inflow = (
    static_params.base_area / env.tau * (h_max - heights_) 
    + static_params.c_d * static_params.orifice_area * jnp.sqrt(2 * static_params.g * heights_)
)
allowed_inflow = jnp.clip(allowed_inflow, 0, 0.2) 

heights, inflows = jnp.meshgrid(heights_, inflows_, indexing="ij")

out_bool = jax.vmap(jax.vmap(recursive_feasibility_parameterized))(heights, inflows)

fig, ax = plt.subplots(1,1, figsize=(6,6), sharey=True)

ax.imshow(out_bool, cmap="plasma", extent=[0, u_max, 0, h_max], interpolation="nearest", origin="lower", aspect='auto')

ax.plot(allowed_inflow, heights_)
ax.set_xlabel(r"$q$")
ax.set_ylabel(r"$h$")

fig.tight_layout()
plt.show()

## Evaluate recursive feasibility of experiment data:

In [ ]:
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results, load_all_experiment_results

cart_pole_data_path = DataPaths().se_cs_experiments / "dmpe" / "fluid_tank"
dmpe_experiment_results = load_all_experiment_results(cart_pole_data_path, model_class=None)

In [ ]:
idx = 1
data_length = 1_000

observations, norm_actions = dmpe_experiment_results[idx]["observations"], dmpe_experiment_results[idx]["actions"]
states = env.vmap_generate_state_from_observation(observations[:data_length])
actions = env.env_properties.action_normalizations.inflow.denormalize(norm_actions)[:data_length]

In [ ]:
heights_ = jnp.linspace(0, h_max, 250)
inflows_ = jnp.linspace(0, u_max, 250)

static_params = env.env_properties.static_params
allowed_inflow = (
    static_params.base_area / env.tau * (h_max - heights_) 
    + static_params.c_d * static_params.orifice_area * jnp.sqrt(2 * static_params.g * heights_)
)
allowed_inflow = jnp.clip(allowed_inflow, 0, 0.2)


heights, inflows = jnp.meshgrid(heights_, inflows_, indexing="ij")

out_bool = jax.vmap(jax.vmap(recursive_feasibility_parameterized))(heights, inflows)

fig, ax = plt.subplots(1,1, figsize=(6,6), sharey=True)
ax.imshow(out_bool, cmap="plasma", extent=[0, u_max, 0, h_max], origin="lower", aspect='auto')
ax.plot(allowed_inflow, heights_)
ax.set_xlabel(r"$q$ in $\frac{\mathrm{m}^3}{\mathrm{s}}$")
ax.set_ylabel(r"$h$ in m")

ax.scatter(actions, states.physical_state.height, s=1, c="r")

fig.tight_layout()
plt.show()

## numerical recursive feasibilty

- Task 1:
    - setup a grid of starting points
    - from each starting point solve an optimization problem with the penalty function as the only arguments
    - See from which starting points you are able to stay within the constraints
    - (repeat with the first action being predetermined for R_XU)
- Task 2:
    - setup a grid of target points
    - from the initial state of the system solve an optimization problem to get as close as possible to each of the target points while staying within the constraints
    - See which of the target points you can actually reach

In [ ]:
from typing import Callable

import exciting_environments as excenvs
from dmpe.models.model_utils import simulate_ahead_with_env
from dmpe.utils.reachability import optimize_actions_multistart

In [ ]:
key = jax.random.PRNGKey(0)
proposed_actions = jax.random.uniform(key=jax.random.PRNGKey(0), shape=(10, 100, 1), minval=-1, maxval=1)

lr = optax.schedules.exponential_decay(
    init_value=1e-1,
    transition_steps=1000,
    transition_begin=0,
    decay_rate=0.1,
    end_value=1e-3,
)

optimizer = optax.adam(lr)

In [ ]:
init_observations = jnp.linspace(-1, 1, 10)[:, None]

observations, _ = simulate_ahead_with_env(
    env,
    init_obs,
    env.generate_state_from_observation(init_obs, env.env_properties),
    proposed_actions[0]
)

plt.plot(observations[:, 0], label="theta")
plt.plot(observations[:, 1], label="omega")
plt.grid(True)
plt.show()

plt.plot(proposed_actions[0])
plt.grid(True)
plt.show()

chosen_actions, loss = optimize_actions_multistart(proposed_actions, init_observations[0], penalty_function, env, optimizer, 200)

observations, _ = simulate_ahead_with_env(
    env,
    init_obs,
    env.generate_state_from_observation(init_obs, env.env_properties),
    chosen_actions    
)

plt.plot(observations[:, 0], label="theta")
plt.plot(observations[:, 1], label="omega")
plt.grid(True)
plt.show()

plt.plot(chosen_actions)
plt.grid(True)
plt.show()